# 1. Introduction
## Feature Engineering

### Objectif

Cette étape consiste à créer des variables explicatives pertinentes à partir des données historiques des commandes.

Le but est d’améliorer les performances des modèles prédictifs en capturant :
- les comportements temporels ;
- les habitudes de commande ;
- les tendances des stations-service.

# 2. Importation des Bibliothèques

In [232]:
import pandas as pd
import numpy as np

from sklearn.preprocessing import LabelEncoder

# Configuration affichage
pd.set_option('display.max_columns', None)
pd.set_option('display.width', 1200)

# 3. Chargement des données propres

In [234]:
df = pd.read_excel("../data/propre/commandes_propres.xlsx")

print(df.shape)

display(df.head())

(34150, 13)


,Date comptabilisation,N° article,N° origine,N° contact origine,Quantité facturée,Code Produit,Type document,NOM CLIENT,Nom sous compte,TYPE,STATUT,Code Station,GROUPE TYPE
0,2020-01-13,A0001,C1234,CC2616,5000,PROD-BLANC,Expédition vente,GIBLAND COMPAGNY,STATION ATTAKE,CODO,Fonctionnelle,ST0049,STATION
1,2020-01-13,A0002,C1234,CC2616,4000,PROD-BLANC,Expédition vente,GIBLAND COMPAGNY,STATION ATTAKE,CODO,Fonctionnelle,ST0049,STATION
2,2020-01-13,A0001,C1249,CC2650,1000,PROD-BLANC,Expédition vente,TONI ET FILS,STATION NDALI,DODO,Fonctionnelle,ST0092,STATION
3,2020-01-13,A0002,C1249,CC2650,16000,PROD-BLANC,Expédition vente,TONI ET FILS,STATION NDALI,DODO,Fonctionnelle,ST0092,STATION
4,2020-01-13,A0001,C1233,CC2614,5000,PROD-BLANC,Expédition vente,AYODELE BUSINESS,STATION OUIDAH,CODO,Fonctionnelle,ST0016,STATION


# 4. Vérification du format des dates
Les modèles temporels ont besoin : de dates exploitables et d’ordre chronologique.

In [236]:
# CONVERSION DES DATES

df["Date comptabilisation"] = pd.to_datetime(
    df["Date comptabilisation"]
)

print(df["Date comptabilisation"].dtype)

datetime64[ns]


In [238]:
# TRI CHRONOLOGIQUE
# Les variables historiques dépendent : de l’ordre temporel et des commandes précédentes.

df = df.sort_values(
    ["Code Station", "Date comptabilisation"]
)

display(df.head())

,Date comptabilisation,N° article,N° origine,N° contact origine,Quantité facturée,Code Produit,Type document,NOM CLIENT,Nom sous compte,TYPE,STATUT,Code Station,GROUPE TYPE
13322,2022-09-27,A0001,C0083,CC6471,20000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13323,2022-09-27,A0002,C0083,CC6471,15000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13637,2022-10-17,A0001,C0083,CC6471,8000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13648,2022-10-17,A0002,C0083,CC6471,10000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION
13727,2022-10-20,A0002,C0083,CC6471,8000,PROD-BLANC,Expédition vente,2M ASSOCIES,STATION BOULEVARD MARINA,CODO,Fonctionnelle,ST0001,STATION


In [240]:
# AGREGER LES COMMANDES JOURNALIERES
# =========================================================

df_daily = (
    df.groupby(
        [
            "Date comptabilisation",
            "Code Station",
            "N° article"
        ]
    )
    .agg({
        "Quantité facturée": "sum"
    })
    .reset_index()
    .copy()
)

In [242]:
all_dates = pd.date_range(
    start=df_daily["Date comptabilisation"].min(),
    end=df_daily["Date comptabilisation"].max(),
    freq="D"
)

In [244]:
df_daily = df_daily.sort_values(
    [
        "Code Station",
        "N° article",
        "Date comptabilisation"
    ]
).copy()

In [258]:
display(df_daily.head(20))

,Date comptabilisation,Code Station,N° article,Quantité facturée
10703,2022-09-27,ST0001,A0001,20000
10959,2022-10-17,ST0001,A0001,8000
11042,2022-10-24,ST0001,A0001,6000
11079,2022-10-25,ST0001,A0001,10000
11143,2022-11-03,ST0001,A0001,15000
11228,2022-11-08,ST0001,A0001,12000
11283,2022-11-11,ST0001,A0001,22000
12009,2023-01-05,ST0001,A0001,10000
12073,2023-01-12,ST0001,A0001,12000
12110,2023-01-16,ST0001,A0001,10000


In [248]:
stations = df_daily["Code Station"].unique()
produits = df_daily["N° article"].unique()

In [250]:
full_index = pd.MultiIndex.from_product(
    [
        all_dates,
        stations,
        produits
    ],
    names=[
        "Date comptabilisation",
        "Code Station",
        "N° article"
    ]
)

In [252]:
df_full = (
    df_daily
    .set_index(
        [
            "Date comptabilisation",
            "Code Station",
            "N° article"
        ]
    )
    .reindex(full_index)
    .reset_index()
)

In [254]:
df_full["Quantité facturée"] = (
    df_full["Quantité facturée"]
    .fillna(0)
)

In [256]:
display(df_full.head(100))

,Date comptabilisation,Code Station,N° article,Quantité facturée
0,2020-01-13,ST0001,A0001,0.0
1,2020-01-13,ST0001,A0002,0.0
2,2020-01-13,ST0002,A0001,0.0
3,2020-01-13,ST0002,A0002,0.0
4,2020-01-13,ST0003,A0001,0.0
...,...,...,...,...
95,2020-01-13,ST0048,A0002,0.0
96,2020-01-13,ST0049,A0001,5000.0
97,2020-01-13,ST0049,A0002,4000.0
98,2020-01-13,ST0050,A0001,0.0


# 5. Création des variables temporelles


In [202]:
#Certains jours génèrent plus ou moins de commandes
df_full["Jour"] = df_full["Date comptabilisation"].dt.day

# Détection de saisonnalité et variations mensuelles
df_full["Mois"] = df_full["Date comptabilisation"].dt.month

# Evolution globale et croissance de l'activité
df_full["Annee"] = df_full["Date comptabilisation"].dt.year

# Les comportement diffèrent lundi, fin de semaine et week-end
df_full["JourSemaine"] = (
    df_full["Date comptabilisation"]
    .dt.dayofweek
)

df_full["Weekend"] = (
    df_full["JourSemaine"]
    .isin([5, 6])
    .astype(int)
)

In [204]:
df_full["FinMois"] = (
    df_full["Date comptabilisation"]
    .dt.is_month_end
    .astype(int)
)

In [206]:
df_full["DebutMois"] = (
    df_full["Date comptabilisation"]
    .dt.is_month_start
    .astype(int)
)

In [208]:
df_full["Trimestre"] = (
    df_full["Date comptabilisation"]
    .dt.quarter
)

# 6. Création des variables historiques

## 6.1 Quantité précédente (Lag_1, Lag_7 et Lag_30)

La dernière commande influence fortement la prochaine commande.

In [210]:
# LAGS TEMPORELS

group_cols = ["Code Station", "N° article"]

df_full["Lag_1"] = (
    df_full.groupby(group_cols)
    ["Quantité facturée"]
    .shift(1)
)

df_full["Lag_7"] = (
    df_full.groupby(group_cols)
    ["Quantité facturée"]
    .shift(7)
)

df_full["Lag_30"] = (
    df_full.groupby(group_cols)
    ["Quantité facturée"]
    .shift(30)
)

## 6.3 Moyenne mobile sur 7 et 30 jours

Capture tendance récente et stabilité des commandes.

In [212]:
# ROLLING MEAN

df_full["Rolling_7j"] = (
    df_full.groupby(group_cols)
    ["Quantité facturée"]
    .transform(
        lambda x: x.rolling(7).mean()
    )
)

In [214]:
# ROLLING MEAN

df_full["Rolling_30j"] = (
    df_full.groupby(group_cols)
    ["Quantité facturée"]
    .transform(
        lambda x: x.rolling(30).mean()
    )
)

## 6.4 Écart-type mobile

Mesure variabilité et instabilité des commandes.

In [216]:
# ROLLING STD

df_full["Rolling_Std_7j"] = (
    df_full.groupby(group_cols)
    ["Quantité facturée"]
    .transform(
        lambda x: x.rolling(7).std()
    )
)


In [218]:

df_full["Rolling_Std_30j"] = (
    df_full.groupby(group_cols)
    ["Quantité facturée"]
    .transform(
        lambda x: x.rolling(30).std()
    )
)

## 7. Encodage des variables catégorielles
Les modèles ML ne comprennent pas le texte.

In [220]:
# ENCODAGE STATIONS

le_station = LabelEncoder()

df_full["StationEncoded"] = le_station.fit_transform(
    df_full["Code Station"]
)

# ENCODAGE PRODUIT transformer catégories en valeures numérique

le_produit = LabelEncoder()

df_full["ProduitEncoded"] = le_produit.fit_transform(
    df_full["N° article"]
)



## 9. Vérification des valeurs manquantes

Les lags créent des NaN

In [224]:
# VALEURS MANQUANTES

print(df_full.isnull().sum())

print(df_full.shape)

Date comptabilisation       0
Code Station                0
N° article                  0
Quantité facturée           0
Jour                        0
Mois                        0
Annee                       0
JourSemaine                 0
Weekend                     0
FinMois                     0
DebutMois                   0
Trimestre                   0
Lag_1                     188
Lag_7                    1316
Lag_30                   5640
Rolling_7j               1128
Rolling_30j              5452
Rolling_Std_7j           1128
Rolling_Std_30j          5452
StationEncoded              0
ProduitEncoded              0
dtype: int64
(409652, 21)


In [226]:
# SUPPRESSION VALEURS MANQUANTES (Les modèles ML ne supportent pas les NaN.)
df_full = df_full.dropna()

In [228]:
# APERCU FINAL

display(df_full.head(20))

print(df_full.shape)

,Date comptabilisation,Code Station,N° article,Quantité facturée,Jour,Mois,Annee,JourSemaine,Weekend,FinMois,DebutMois,Trimestre,Lag_1,Lag_7,Lag_30,Rolling_7j,Rolling_30j,Rolling_Std_7j,Rolling_Std_30j,StationEncoded,ProduitEncoded
5640,2020-02-12,ST0001,A0001,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0,0
5641,2020-02-12,ST0001,A0002,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,0,1
5642,2020-02-12,ST0002,A0001,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,1,0
5643,2020-02-12,ST0002,A0002,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,1,1
5644,2020-02-12,ST0003,A0001,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,2,0
5645,2020-02-12,ST0003,A0002,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,2,1
5646,2020-02-12,ST0004,A0001,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,3,0
5647,2020-02-12,ST0004,A0002,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,3,1
5648,2020-02-12,ST0005,A0001,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,4,0
5649,2020-02-12,ST0005,A0002,0.0,12,2,2020,2,0,0,0,1,0.0,0.0,0.0,0.000000,0.000000,0.000000,0.000000,4,1


(404012, 21)


## 10. Sélection des variables ML

C'est une étape est importante où je définis ce que le modèle verra et ce qu’il devra prédire.

In [49]:
# VARIABLES DU MODELE

features = [
    "StationEncoded",
    "ProduitEncoded",
    "Jour",
    "Mois",
    "Annee",
    "JourSemaine",
    "Weekend",
    "Lag_1",
    "Lag_7",
    "Rolling_7j",
    "Rolling_Std_7j"
]

target = "Quantité facturée"

print(features)

['StationEncoded', 'ProduitEncoded', 'Jour', 'Mois', 'Annee', 'JourSemaine', 'Weekend', 'Lag_1', 'Lag_7', 'Rolling_7j', 'Rolling_Std_7j']


## 11. Sauvegarde du dataset final

In [52]:
# SAUVEGARDE

df.to_excel(
    "../data/propre/dataset_ml.xlsx",
    index=False
)

print("Dataset ML sauvegardé.")

Dataset ML sauvegardé.
